In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from raptor import Tensor


In [2]:
x = Tensor([1., 2., 3.])
y = x.sum()
y.backward()
print(x.grad)


[1. 1. 1.]


In [11]:
from raptor import Tensor

x = Tensor(2.0)
y = Tensor(3.0)
z = x * y + x

print(type(z))
print(z)

z.backward()


print("x.grad:", x.grad)
print("y.grad:", y.grad)


<class 'raptor.engine.Tensor'>
Tensor(data = 8.0, grad =0.0)
x.grad: 4.0
y.grad: 2.0


In [3]:
import numpy as np
from raptor import Tensor

a = Tensor(np.array([[1., 2., 3.],
                     [4., 5., 6.]], dtype=np.float32))
b = Tensor(np.array([10., 20., 30.], dtype=np.float32))

c = a + b
loss = c.sum()
loss.backward()

print("a.grad:")
print(a.grad)

print("b.grad:")
print(b.grad)


a.grad:
[[1. 1. 1.]
 [1. 1. 1.]]
b.grad:
[2. 2. 2.]


In [4]:
x = Tensor(np.array([1., 2., 3., 4.], dtype=np.float32))
y = x.mean()
y.backward()

print("y.data:", y.data)
print("x.grad:", x.grad)


y.data: 2.5
x.grad: [0.25 0.25 0.25 0.25]


In [4]:
from raptor import Tensor
from raptor.ops import relu
import numpy as np

x = Tensor(np.array([-2., -1., 0., 1., 2.], dtype=np.float32))
y = relu(x)
z = y.sum()
z.backward()

print("y.data:", y.data)
print("x.grad:", x.grad)


y.data: [0. 0. 0. 1. 2.]
x.grad: [0. 0. 0. 1. 1.]


In [5]:
import numpy as np
from raptor.ops import tanh

x = Tensor(np.array([0.], dtype=np.float32))
y = tanh(x)
y.backward()

print("y.data:", y.data)
print("x.grad:", x.grad)


y.data: [0.]
x.grad: [1.]


In [6]:
from raptor.ops import sigmoid

x = Tensor(np.array([0.], dtype=np.float32))
y = sigmoid(x)
y.backward()

print("y.data:", y.data)
print("x.grad:", x.grad)


y.data: [0.5]
x.grad: [0.25]


In [2]:
import numpy as np
from raptor import Tensor
from raptor import nn

model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = Tensor(np.random.randn(3, 4).astype(np.float32))
out = model(x)

print("out shape:", out.data.shape)
print("num params:", len(model.parameters()))


out shape: (3, 2)
num params: 4


In [3]:
loss = out.sum()
loss.backward()

for i, p in enumerate(model.parameters()):
    print(i, p.data.shape, p.grad.shape)


0 (8, 4) (8, 4)
1 (8,) (8,)
2 (2, 8) (2, 8)
3 (2,) (2,)


In [2]:
import numpy as np
from raptor import Tensor
from raptor import nn

pred = Tensor(np.array([1.0, 2.0, 3.0], dtype=np.float32))
target = Tensor(np.array([1.5, 2.5, 3.5], dtype=np.float32), requires_grad=False)

loss = nn.MSELoss()(pred, target)
print("loss:", loss.data)

loss.backward()
print("pred.grad:", pred.grad)


loss: 0.25
pred.grad: [-0.33333334 -0.33333334 -0.33333334]


In [3]:
import numpy as np
from raptor import Tensor
from raptor import nn

logits = Tensor(np.array([[2.0, 1.0, 0.1]], dtype=np.float32))
target = np.array([0])

loss = nn.CrossEntropyLoss()(logits, target)
print("loss:", loss.data)

loss.backward()
print("logits.grad:", logits.grad)


loss: 0.41703007
logits.grad: [[-0.3409989   0.24243295  0.0985659 ]]


In [ ]:
import numpy as np
from raptor import Tensor
from raptor.optim import SGD

w = Tensor(np.array([1.0], dtype=np.float32))
w.grad = np.array([0.5], dtype=np.float32)

opt = SGD([w], lr=0.1)
opt.step()

print(w.data)


[0.95]


In [6]:
import numpy as np
from raptor import Tensor
from raptor.optim import Adam

w = Tensor(np.array([1.0], dtype=np.float32))
w.grad = np.array([0.5], dtype=np.float32)

opt = Adam([w], lr=0.1)
opt.step()

print(w.data)


[0.9]


In [3]:
import numpy as np

from raptor import Tensor, nn
from raptorgraph.server import set_current_tensor

x = Tensor(np.array([[1.0, -2.0, 3.0, 0.5]], dtype=np.float32), requires_grad=False)

model = nn.Sequential(
    nn.Linear(4, 6),
    nn.ReLU(),
    nn.Linear(6, 3),
)

labels = np.array([2], dtype=np.int64)

logits = model(x)
loss = nn.CrossEntropyLoss()(logits, labels)
loss.backward()

set_current_tensor(loss, name="quick_test_graph")


{'name': 'quick_test_graph',
 'node_count': 13,
 'edge_count': 12,
 'nodes': [{'id': '3000021538000',
   'op': 'cross_entropy',
   'shape': [],
   'data': {'kind': 'full', 'values': [0.32723933458328247]},
   'grad': {'kind': 'full', 'values': [1.0]},
   'requires_grad': True},
  {'id': '3000021322320',
   'op': '+',
   'shape': [1, 3],
   'data': {'kind': 'full',
    'values': [-6.140043258666992, -3.486985445022583, -2.4699337482452393]},
   'grad': {'kind': 'full',
    'values': [0.018364261835813522, 0.2607244849205017, -0.2790887951850891]},
   'requires_grad': True},
  {'id': '3000021332816',
   'op': '@',
   'shape': [1, 3],
   'data': {'kind': 'full',
    'values': [-6.140043258666992, -3.486985445022583, -2.4699337482452393]},
   'grad': {'kind': 'full',
    'values': [0.018364261835813522, 0.2607244849205017, -0.2790887951850891]},
   'requires_grad': True},
  {'id': '3000021222464',
   'op': 'relu',
   'shape': [1, 6],
   'data': {'kind': 'full',
    'values': [2.58168721199

In [5]:
import numpy as np

from raptor import Tensor, nn
from raptorgraph.client import register_graph

x = Tensor(np.array([[1.0, -2.0, 3.0, 0.5]], dtype=np.float32), requires_grad=False)

model = nn.Sequential(
    nn.Linear(4, 6),
    nn.ReLU(),
    nn.Linear(6, 3),
)

labels = np.array([2], dtype=np.int64)

logits = model(x)
loss = nn.CrossEntropyLoss()(logits, labels)
loss.backward()

register_graph(loss, name="mygraph")


{'name': 'mygraph',
 'node_count': 13,
 'edge_count': 12,
 'nodes': [{'id': '3000025403216',
   'op': 'cross_entropy',
   'shape': [],
   'data': {'kind': 'full', 'values': [6.050516128540039]},
   'grad': {'kind': 'full', 'values': [1.0]},
   'requires_grad': True},
  {'id': '3000025399376',
   'op': '+',
   'shape': [1, 3],
   'data': {'kind': 'full',
    'values': [2.5681674480438232, -4.325527667999268, -3.478975534439087]},
   'grad': {'kind': 'full',
    'values': [0.9966326355934143, 0.00101074471604079, -0.9976433515548706]},
   'requires_grad': True},
  {'id': '3000024458064',
   'op': 'input',
   'shape': [3],
   'data': {'kind': 'full', 'values': [0.0, 0.0, 0.0]},
   'grad': {'kind': 'full',
    'values': [0.9966326355934143, 0.00101074471604079, -0.9976433515548706]},
   'requires_grad': True},
  {'id': '3000025402832',
   'op': '@',
   'shape': [1, 3],
   'data': {'kind': 'full',
    'values': [2.5681674480438232, -4.325527667999268, -3.478975534439087]},
   'grad': {'kind